# Explorando padrões com clusterização não supervisionada

##  Contexto geral da etapa

Nesta etapa do projeto, será aplicada uma abordagem de **aprendizado não supervisionado** com o objetivo de identificar padrões naturais presentes nos dados, sem o uso de rótulos pré-definidos.

A ideia central aqui é investigar se existem **grupos latentes (clusters)** que emergem a partir das características das amostras, permitindo uma compreensão mais estruturada do comportamento dos dados.

---

## Abordagem adotada

Para essa tarefa, será utilizado o algoritmo de **K-Means**, um dos métodos mais clássicos e eficientes de clusterização.

O K-Means será responsável por:

* Agrupar os dados com base em similaridade
* Minimizar a variância intra-cluster
* Maximizar a separação entre grupos distintos

---

##  Preparação dos dados

Antes da aplicação do algoritmo, os dados passam por etapas fundamentais de pré-processamento, incluindo:

* Padronização das variáveis
* Redução de dimensionalidade com **PCA (Principal Component Analysis)**

Essa etapa é necessária porque o conjunto atual possui múltiplas variáveis, o que pode:

* Dificultar a separação natural dos grupos
* Introduzir redundância entre features
* Aumentar a complexidade do espaço de decisão

---

## Uso do PCA

O PCA será utilizado para transformar o espaço original em um novo conjunto de variáveis (componentes principais), preservando a maior parte possível da variância dos dados.

Isso permite:

* Reduzir ruído e redundância
* Melhorar a definição de distância entre pontos
* Facilitar a formação de clusters mais consistentes

---

##  Objetivo da análise

Após a clusterização, os resultados serão:

* Analisados qualitativamente para interpretação dos grupos formados
* Comparados com abordagens de aprendizado supervisionado previamente aplicadas
* Avaliados quanto à capacidade de capturar padrões semelhantes aos modelos com rótulos

---

##  Finalidade

O objetivo final desta etapa é verificar se uma abordagem não supervisionada consegue:

* Revelar estruturas ocultas nos dados
* Aproximar-se de padrões identificados em modelos supervisionados
* Oferecer uma alternativa interpretável e eficiente para segmentação dos dados




In [3]:
# IMPORTAÇÃO DE BIBLIOTECAS
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os
import seaborn as sns
from pathlib import Path
import warnings
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

warnings.filterwarnings("ignore")

SEED = 42


# DETECÇÃO DE AMBIENTE
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


# CARREGAMENTO DO DATASET
if IN_COLAB:
    print("Ambiente Google Colab detectado.")

    drive.mount('/content/drive')

    DATA_PATH = Path(
        "/content/drive/MyDrive/EDA_AquaSense/Dataset/raw/Combined Data/Combined_dataset.csv"
    )

else:
    print("Ambiente local/VS Code detectado.")

    DATA_PATH = Path("../../dataset/Combined_dataset.csv")


df = pd.read_csv(DATA_PATH)

print("Dataset carregado com sucesso.")
print(f"Shape do dataset: {df.shape}")

df.head()

Ambiente Google Colab detectado.
Mounted at /content/drive
Dataset carregado com sucesso.
Shape do dataset: (2827977, 14)


,Country,Area,Waterbody Type,Date,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),Nitrogen (mg/l),Nitrate (mg/l),CCME_Values,CCME_WQI
0,Canada,SE649035-145565,River,12-01-1974,0.059248,1.30,8.1500,0.011917,8.07500,9.885,0.343917,11.73155,100.0,Excellent
1,Canada,SE649035-145565,River,12-01-1975,0.039821,1.38,7.8000,0.009417,7.73333,10.150,0.449083,11.82009,100.0,Excellent
2,Canada,SE649035-145565,River,12-01-1976,0.031341,2.23,7.8000,0.011000,7.46667,10.235,0.220750,14.87472,100.0,Excellent
3,Canada,SE649035-145565,River,12-01-1977,0.020501,1.61,8.1500,0.012333,7.78333,11.116,0.572250,15.89293,100.0,Excellent
4,Canada,SE649035-145565,River,12-01-1978,0.020023,1.64,4.3708,0.006182,7.10000,7.068,0.371091,15.22888,100.0,Excellent


In [4]:
if IN_COLAB:
    parquet_path = Path(
        "/content/drive/MyDrive/EDA_AquaSense/Dataset/processed/water_quality_clusters.parquet"
    )
else:
    parquet_path = Path(
        "../../dataset/processed/water_quality_clusters.parquet"
    )

df.to_parquet(parquet_path, index=False)

print("Dataset convertido para Parquet com sucesso.")
print(f"Arquivo salvo em: {parquet_path}")

Dataset convertido para Parquet com sucesso.
Arquivo salvo em: /content/drive/MyDrive/EDA_AquaSense/Dataset/processed/water_quality_clusters.parquet


In [5]:
df = pd.read_parquet(parquet_path)

print("Dataset Parquet carregado com sucesso.")
print(f"Shape do dataset: {df.shape}")

df.head()

Dataset Parquet carregado com sucesso.
Shape do dataset: (2827977, 14)


,Country,Area,Waterbody Type,Date,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),Nitrogen (mg/l),Nitrate (mg/l),CCME_Values,CCME_WQI
0,Canada,SE649035-145565,River,12-01-1974,0.059248,1.30,8.1500,0.011917,8.07500,9.885,0.343917,11.73155,100.0,Excellent
1,Canada,SE649035-145565,River,12-01-1975,0.039821,1.38,7.8000,0.009417,7.73333,10.150,0.449083,11.82009,100.0,Excellent
2,Canada,SE649035-145565,River,12-01-1976,0.031341,2.23,7.8000,0.011000,7.46667,10.235,0.220750,14.87472,100.0,Excellent
3,Canada,SE649035-145565,River,12-01-1977,0.020501,1.61,8.1500,0.012333,7.78333,11.116,0.572250,15.89293,100.0,Excellent
4,Canada,SE649035-145565,River,12-01-1978,0.020023,1.64,4.3708,0.006182,7.10000,7.068,0.371091,15.22888,100.0,Excellent


In [6]:
df_amostra = (
    df.groupby("Country", group_keys=False)
      .sample(frac=0.05, random_state=42)
      .reset_index(drop=True)
)

In [7]:
print(df_amostra.shape)

(141399, 14)


In [8]:
df["Country"].value_counts(normalize=True)
df_amostra["Country"].value_counts(normalize=True)


,proportion
Country,
England,0.752905
USA,0.146331
Ireland,0.083105
China,0.016266
Canada,0.001393


In [9]:
print("Original:")
print(df["Country"].value_counts(normalize=True))

print("\nAmostra:")
print(df_amostra["Country"].value_counts(normalize=True))

Original:
Country
England    0.752905
USA        0.146329
Ireland    0.083105
China      0.016265
Canada     0.001396
Name: proportion, dtype: float64

Amostra:
Country
England    0.752905
USA        0.146331
Ireland    0.083105
China      0.016266
Canada     0.001393
Name: proportion, dtype: float64


In [10]:
print("Original:")
print(df["CCME_Values"].describe())

print("\nAmostra:")
print(df_amostra["CCME_Values"].describe())

Original:
count    2.827977e+06
mean     8.504668e+01
std      1.764665e+01
min      3.130414e+01
25%      7.715349e+01
50%      9.059609e+01
75%      1.000000e+02
max      1.000000e+02
Name: CCME_Values, dtype: float64

Amostra:
count    141399.000000
mean         85.038443
std          17.650268
min          34.607162
25%          77.230685
50%          90.543682
75%         100.000000
max         100.000000
Name: CCME_Values, dtype: float64


In [11]:
df_amostra.to_parquet(
    "/content/drive/MyDrive/EDA_AquaSense/amostra_clusters.parquet",
    index=False
)

In [12]:
df = pd.read_parquet(
    "/content/drive/MyDrive/EDA_AquaSense/amostra_clusters.parquet"
)

print(df.shape)

df.head()

(141399, 14)


,Country,Area,Waterbody Type,Date,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),Nitrogen (mg/l),Nitrate (mg/l),CCME_Values,CCME_WQI
0,Canada,CZPOH_1117,River,05-03-2012,0.092790,2.25455,9.43636,0.06100,7.50000,9.40000,0.400,6.595930,93.116725,Good
1,Canada,FISW_32,Lake,02-12-2003,0.043792,2.13333,9.82400,0.00200,7.79000,12.00000,0.230,2.355164,100.000000,Excellent
2,Canada,CZPOH_1036,River,12-03-2018,0.085920,2.01667,11.78333,0.02568,7.36667,8.91667,0.400,7.673120,100.000000,Excellent
3,Canada,IEEA_10_32,Lake,08-06-2001,0.015920,0.55000,9.82400,0.00400,7.79000,16.80000,0.007,0.624207,100.000000,Excellent
4,Canada,ES0910524,River,11-09-2012,0.150000,2.13333,10.32500,0.07250,7.79000,8.32500,0.400,1.775000,92.882835,Good


In [13]:
df.head()

,Country,Area,Waterbody Type,Date,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),Nitrogen (mg/l),Nitrate (mg/l),CCME_Values,CCME_WQI
0,Canada,CZPOH_1117,River,05-03-2012,0.092790,2.25455,9.43636,0.06100,7.50000,9.40000,0.400,6.595930,93.116725,Good
1,Canada,FISW_32,Lake,02-12-2003,0.043792,2.13333,9.82400,0.00200,7.79000,12.00000,0.230,2.355164,100.000000,Excellent
2,Canada,CZPOH_1036,River,12-03-2018,0.085920,2.01667,11.78333,0.02568,7.36667,8.91667,0.400,7.673120,100.000000,Excellent
3,Canada,IEEA_10_32,Lake,08-06-2001,0.015920,0.55000,9.82400,0.00400,7.79000,16.80000,0.007,0.624207,100.000000,Excellent
4,Canada,ES0910524,River,11-09-2012,0.150000,2.13333,10.32500,0.07250,7.79000,8.32500,0.400,1.775000,92.882835,Good


In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 141399 entries, 0 to 141398
Data columns (total 14 columns):
 #   Column                            Non-Null Count   Dtype  
---  ------                            --------------   -----  
 0   Country                           141399 non-null  object 
 1   Area                              141399 non-null  object 
 2   Waterbody Type                    141399 non-null  object 
 3   Date                              141399 non-null  object 
 4   Ammonia (mg/l)                    141399 non-null  float64
 5   Biochemical Oxygen Demand (mg/l)  141399 non-null  float64
 6   Dissolved Oxygen (mg/l)           141399 non-null  float64
 7   Orthophosphate (mg/l)             141399 non-null  float64
 8   pH (ph units)                     141399 non-null  float64
 9   Temperature (cel)                 141399 non-null  float64
 10  Nitrogen (mg/l)                   141399 non-null  float64
 11  Nitrate (mg/l)                    141399 non-null  f

In [15]:
features = [
    "Ammonia (mg/l)",
    "Biochemical Oxygen Demand (mg/l)",
    "Dissolved Oxygen (mg/l)",
    "Orthophosphate (mg/l)",
    "pH (ph units)",
    "Temperature (cel)",
    "Nitrogen (mg/l)",
    "Nitrate (mg/l)"
]

In [17]:
X = df[features]

In [18]:
X.head()

,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),Nitrogen (mg/l),Nitrate (mg/l)
0,0.092790,2.25455,9.43636,0.06100,7.50000,9.40000,0.400,6.595930
1,0.043792,2.13333,9.82400,0.00200,7.79000,12.00000,0.230,2.355164
2,0.085920,2.01667,11.78333,0.02568,7.36667,8.91667,0.400,7.673120
3,0.015920,0.55000,9.82400,0.00400,7.79000,16.80000,0.007,0.624207
4,0.150000,2.13333,10.32500,0.07250,7.79000,8.32500,0.400,1.775000


In [20]:
X.shape

(141399, 8)

## Padronização dos dados

In [21]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

In [22]:
X_scaled[:5]

array([[-0.19638328, -0.1602649 , -0.30692906, -0.31355503, -0.48765257,
        -0.49160691, -0.77489944,  0.30540414],
       [-0.20550321, -0.16789009, -0.09781613, -0.3418229 ,  0.10631135,
         0.02913367, -0.80230366, -0.40223447],
       [-0.19766199, -0.17522844,  0.95914712, -0.33047743, -0.7607326 ,
        -0.58841058, -0.77489944,  0.48515025],
       [-0.21069105, -0.26748745, -0.09781613, -0.34086467,  0.10631135,
         0.99050091, -0.83825156, -0.69107191],
       [-0.18573487, -0.16789009,  0.17244901, -0.30804519,  0.10631135,
        -0.70691311, -0.77489944, -0.49904397]])

In [23]:
X_scaled = pd.DataFrame(
    X_scaled,
    columns=features
)

In [24]:
X_scaled.head()

,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),Nitrogen (mg/l),Nitrate (mg/l)
0,-0.196383,-0.160265,-0.306929,-0.313555,-0.487653,-0.491607,-0.774899,0.305404
1,-0.205503,-0.167890,-0.097816,-0.341823,0.106311,0.029134,-0.802304,-0.402234
2,-0.197662,-0.175228,0.959147,-0.330477,-0.760733,-0.588411,-0.774899,0.485150
3,-0.210691,-0.267487,-0.097816,-0.340865,0.106311,0.990501,-0.838252,-0.691072
4,-0.185735,-0.167890,0.172449,-0.308045,0.106311,-0.706913,-0.774899,-0.499044


## Redução de dimensionalidade com PCA

Após a padronização dos dados, o conjunto passou a ser representado por **8 dimensões**, ou seja, 8 variáveis numéricas utilizadas na análise de agrupamento.

Esse número de dimensões impacta diretamente a clusterização porque algoritmos como o **K-Means** calculam a similaridade entre os registros com base em distâncias. Quando existem muitas dimensões, essas distâncias podem se tornar menos intuitivas e mais difíceis de interpretar, fenômeno conhecido como **maldição da dimensionalidade**.

Em espaços com muitas variáveis, os pontos podem parecer igualmente distantes entre si, dificultando a identificação de grupos naturais. Além disso, algumas variáveis podem conter informações redundantes ou ruído, o que pode prejudicar a formação dos clusters.

Por isso, aplicamos o `PCA (Principal Component Analysis)`. O PCA transforma as variáveis originais em novos eixos chamados **componentes principais**, que concentram a maior parte da variância dos dados. Dessa forma, conseguimos reduzir a dimensionalidade mantendo o máximo possível de informação relevante.

No contexto desta análise, o PCA será utilizado para:

- reduzir a complexidade dos dados;
- melhorar a visualização dos agrupamentos;
- diminuir possíveis redundâncias entre variáveis;
- facilitar a interpretação dos padrões;
- apoiar a escolha e avaliação dos clusters.

Assim, antes de aplicar definitivamente o algoritmo de clusterização, analisamos quantos componentes principais são suficientes para representar bem os dados.

In [25]:
from sklearn.decomposition import PCA

pca = PCA()
X_pca = pca.fit_transform(X_scaled)

In [26]:
import numpy as np
import plotly.express as px
import pandas as pd

# Variância explicada
explained_variance = pca.explained_variance_ratio_

# Variância acumulada
cumulative_variance = np.cumsum(explained_variance)

# Exibir valores
for i, var in enumerate(explained_variance):
    print(f'Componente {i+1}: {var:.4f}')

print("\nVariância acumulada:")
print(cumulative_variance)

# DataFrame para o gráfico
pca_variance_df = pd.DataFrame({
    'Componente': range(1, len(cumulative_variance)+1),
    'Variancia_Acumulada': cumulative_variance
})

# Gráfico
fig = px.line(
    pca_variance_df,
    x='Componente',
    y='Variancia_Acumulada',
    markers=True,
    title='Variância Explicada Acumulada - PCA'
)

fig.update_layout(
    xaxis_title='Número de Componentes',
    yaxis_title='Variância Acumulada',
    template='plotly_white'
)

fig.show()

Componente 1: 0.2429
Componente 2: 0.1782
Componente 3: 0.1549
Componente 4: 0.1277
Componente 5: 0.0957
Componente 6: 0.0876
Componente 7: 0.0698
Componente 8: 0.0432

Variância acumulada:
[0.2429054  0.42113101 0.57599755 0.70367407 0.79934114 0.8869721
 0.95680266 1.        ]


## Análise da variância explicada acumulada no PCA



A análise da variância explicada acumulada foi utilizada para entender quantos componentes principais seriam necessários para preservar a maior parte da estrutura dos dados.

No conjunto analisado, os componentes apresentaram aproximadamente os seguintes percentuais de variância explicada:

| Componente | Variância explicada aproximada |
|---|---|
| PC1 | 24% |
| PC2 | 17% |
| PC3 | 15% |
| PC4 | 12% |
| PC5 | 9% |
| PC6 | 8% |
| PC7 | 6% |
| PC8 | 4% |

A variância acumulada indicou que:

| Componentes mantidos | Variância acumulada aproximada |
|---|---|
| PC1 | 24% |
| PC1 + PC2 | 42% |
| PC1 + PC2 + PC3 | 57% |
| PC1 + PC2 + PC3 + PC4 | 70% |
| PC1 até PC5 | 79% |
| PC1 até PC6 | 88% |
| PC1 até PC7 | 95% |

Esse resultado mostra que os dados ambientais não possuem apenas um único padrão dominante. Se existisse um único padrão muito forte, o primeiro componente explicaria uma parcela muito maior da variância total. No entanto, como a variância está distribuída entre vários componentes, isso indica que a qualidade da água é influenciada por diferentes combinações de parâmetros físico-químicos.

Esse comportamento é coerente com dados ambientais reais, pois variáveis como amônia, DBO, oxigênio dissolvido, nitrato, pH, temperatura, nitrogênio e ortofosfato podem se relacionar de formas diferentes dependendo do tipo de impacto ambiental.

Por isso, optou-se por manter 6 componentes principais, preservando aproximadamente 88% da variabilidade dos dados. Essa escolha representa um equilíbrio entre manter a maior parte da informação relevante e reduzir a complexidade do conjunto original.

Na prática, isso significa que o modelo deixou de trabalhar diretamente com as 8 variáveis originais e passou a trabalhar com 6 componentes que resumem os principais padrões ambientais presentes nos dados.

In [27]:
pca = PCA(n_components=6)

X_pca = pca.fit_transform(X_scaled)

print(X_pca.shape)

(141399, 6)


In [28]:
from sklearn.cluster import KMeans
import plotly.express as px
import pandas as pd

wcss = []

K_range = range(1, 11)

for k in K_range:
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    kmeans.fit(X_pca)

    wcss.append(kmeans.inertia_)

# DataFrame
elbow_df = pd.DataFrame({
    'Clusters': list(K_range),
    'WCSS': wcss
})

# Gráfico
fig = px.line(
    elbow_df,
    x='Clusters',
    y='WCSS',
    markers=True,
    title='Elbow Method'
)

fig.update_layout(
    xaxis_title='Número de Clusters (K)',
    yaxis_title='WCSS',
    template='plotly_white'
)

fig.show()

In [30]:
from sklearn.metrics import silhouette_score

silhouette_scores = []

K_range = range(2, 7)

for k in K_range:

    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    cluster_labels = kmeans.fit_predict(X_pca)

    score = silhouette_score(X_pca, cluster_labels)

    silhouette_scores.append(score)

# DataFrame
silhouette_df = pd.DataFrame({
    'Clusters': list(K_range),
    'Silhouette Score': silhouette_scores
})

# Gráfico
fig = px.line(
    silhouette_df,
    x='Clusters',
    y='Silhouette Score',
    markers=True,
    title='Silhouette Score'
)

fig.update_layout(
    xaxis_title='Número de Clusters (K)',
    yaxis_title='Silhouette Score',
    template='plotly_white'
)

fig.show()

## Definição da quantidade de clusters

Após a aplicação do PCA, foi necessário definir quantos grupos o K-Means deveria formar. Para isso, foram utilizados dois métodos complementares: o Elbow Method e o Silhouette Score.

O Elbow Method avalia o WCSS, que representa a soma das distâncias dos pontos até o centroide do cluster ao qual pertencem. Em termos simples, essa métrica indica o quanto as amostras ainda estão espalhadas dentro dos grupos.

Quando existe apenas 1 cluster, todas as amostras são colocadas no mesmo grupo. Isso gera um WCSS muito alto, pois amostras com comportamentos físico-químicos muito diferentes ficam misturadas. À medida que o número de clusters aumenta, o algoritmo consegue separar melhor as amostras, reduzindo o WCSS.

No gráfico do Elbow Method, observou-se uma queda mais acentuada entre K=1 e K=4. Depois disso, a redução do WCSS passou a ser menor. Isso indica que, a partir de aproximadamente 3 ou 4 clusters, adicionar novos grupos já não melhora tanto a compactação dos agrupamentos.

Entretanto, o Elbow Method sozinho não é suficiente para definir a melhor quantidade de clusters. Por isso, também foi utilizado o Silhouette Score.

O Silhouette Score mede duas coisas ao mesmo tempo:

- se as amostras estão parecidas com o próprio cluster;
- se estão bem separadas dos outros clusters.

Os resultados foram aproximadamente:

| Número de clusters | Silhouette Score |
|---|---|
| 2 | 0.49 |
| 3 | 0.45 |
| 4 | 0.33 |
| 5 | 0.18 |
| 6 | 0.21 |

O maior valor foi obtido com K=2, indicando que dois grupos produzem a separação matemática mais forte. No entanto, K=3 ainda apresentou um valor bastante próximo e manteve boa qualidade de separação.

A escolha por 3 clusters foi feita porque, além de apresentar um bom resultado no Silhouette Score, ela também faz mais sentido para a interpretação ambiental do problema. Com 3 grupos, é possível representar uma gradação da qualidade da água:

- condição adequada;
- condição de atenção;
- condição crítica.

Essa escolha também se conecta diretamente com a classificação utilizada no modelo supervisionado, que separa as amostras em Adequada, Atenção e Crítica.

Portanto, a decisão por K=3 não foi baseada apenas na métrica matemática, mas também na coerência ambiental e na interpretabilidade dos resultados.

In [31]:
kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

clusters = kmeans.fit_predict(X_pca)

In [33]:
df['Cluster'] = clusters

In [34]:
df.head()

,Country,Area,Waterbody Type,Date,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),Nitrogen (mg/l),Nitrate (mg/l),CCME_Values,CCME_WQI,Cluster
0,Canada,CZPOH_1117,River,05-03-2012,0.092790,2.25455,9.43636,0.06100,7.50000,9.40000,0.400,6.595930,93.116725,Good,1
1,Canada,FISW_32,Lake,02-12-2003,0.043792,2.13333,9.82400,0.00200,7.79000,12.00000,0.230,2.355164,100.000000,Excellent,1
2,Canada,CZPOH_1036,River,12-03-2018,0.085920,2.01667,11.78333,0.02568,7.36667,8.91667,0.400,7.673120,100.000000,Excellent,1
3,Canada,IEEA_10_32,Lake,08-06-2001,0.015920,0.55000,9.82400,0.00400,7.79000,16.80000,0.007,0.624207,100.000000,Excellent,1
4,Canada,ES0910524,River,11-09-2012,0.150000,2.13333,10.32500,0.07250,7.79000,8.32500,0.400,1.775000,92.882835,Good,1


In [40]:
features = [
    'Ammonia (mg/l)',
    'Biochemical Oxygen Demand (mg/l)',
    'Dissolved Oxygen (mg/l)',
    'Orthophosphate (mg/l)',
    'pH (ph units)',
    'Temperature (cel)',
    'Nitrogen (mg/l)',
    'Nitrate (mg/l)'
]

for cluster in sorted(df['Cluster'].unique()):

    print(f'\nCLUSTER {cluster} \n')

    cluster_data = df[df['Cluster'] == cluster][features]

    stats = cluster_data.describe().round(2)

    display(stats)


CLUSTER 0 



,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),Nitrogen (mg/l),Nitrate (mg/l)
count,15812.00,15812.00,15812.00,15812.00,15812.00,15812.00,15812.00,15812.00
mean,1.97,5.96,10.07,4.33,7.59,12.58,18.16,13.68
std,3.72,6.51,1.29,4.32,0.42,4.04,8.59,12.33
min,0.01,0.50,0.04,0.00,0.53,0.00,0.08,0.10
25%,0.18,2.70,10.20,0.78,7.40,10.30,13.30,4.50
50%,0.50,3.50,10.20,4.04,7.68,11.46,17.30,12.20
75%,2.03,7.10,10.20,6.42,7.80,15.40,23.20,18.30
max,41.20,93.00,18.90,90.00,12.60,70.80,46.00,153.00



CLUSTER 1 



,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),Nitrogen (mg/l),Nitrate (mg/l)
count,123446.00,123446.00,123446.00,123446.00,123446.00,123446.00,123446.00,123446.00
mean,0.52,3.00,10.00,0.22,7.76,11.75,3.57,3.65
std,2.17,4.44,1.92,0.51,0.49,5.11,3.17,3.14
min,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
25%,0.03,1.50,9.73,0.03,7.60,8.75,0.52,0.90
50%,0.04,2.05,10.20,0.07,7.80,11.46,3.01,3.73
75%,0.15,2.70,11.10,0.14,8.00,14.10,5.00,4.50
max,34.50,98.00,20.00,8.20,29.20,96.00,27.40,32.00



CLUSTER 2 



,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),Nitrogen (mg/l),Nitrate (mg/l)
count,2141.00,2141.00,2141.00,2141.00,2141.00,2141.00,2141.00,2141.00
mean,31.06,100.22,10.13,2.81,7.60,12.61,3.98,3.43
std,24.72,77.08,1.17,4.40,0.68,3.91,4.57,4.51
min,0.00,0.95,0.12,0.00,2.10,0.90,0.00,0.00
25%,15.80,25.00,10.20,0.14,7.42,11.46,1.00,0.90
50%,29.60,91.00,10.20,0.45,7.75,11.46,3.99,4.50
75%,40.00,166.00,10.20,4.50,7.78,13.60,5.00,4.50
max,194.00,255.00,15.40,72.00,13.00,58.90,41.60,120.00


## Criação dos clusters e análise estatística dos grupos

Após a definição de três clusters, o algoritmo K-Means foi aplicado sobre os dados transformados pelo PCA. Essa etapa teve como objetivo agrupar as amostras de acordo com a similaridade dos seus perfis físico-químicos.

É importante destacar que o K-Means não utilizou os rótulos do modelo supervisionado. Ou seja, o algoritmo não sabia previamente quais amostras eram classificadas como `Adequada`, `Atenção` ou `Crítica`. Ele agrupou os dados apenas com base nas distâncias entre as amostras no espaço dos componentes principais.

Mesmo assim, os agrupamentos formados apresentaram forte coerência ambiental, permitindo a seguinte interpretação:

| Cluster | Interpretação ambiental |
|---|---|
| Cluster 1 | Adequada — água mais preservada |
| Cluster 0 | Atenção — condição intermediária |
| Cluster 2 | Crítica — forte degradação ambiental |

Para compreender melhor o comportamento de cada grupo, foram analisadas estatísticas descritivas como média, mediana, desvio padrão, quartis, valores mínimos e máximos. Essa etapa foi essencial porque a média sozinha não é suficiente para representar bem dados ambientais, principalmente quando existem assimetrias e valores extremos.

O Cluster 1 concentrou a maior parte das amostras e apresentou o perfil mais estável. Os valores de amônia, DBO, nitrato, nitrogênio e ortofosfato foram baixos, enquanto o oxigênio dissolvido permaneceu elevado. Além disso, as métricas mostraram menor dispersão, indicando que as amostras desse grupo são mais parecidas entre si. Esse comportamento reforça a interpretação de que o Cluster 1 representa águas em condição mais preservada.

O Cluster 0 apresentou um comportamento intermediário. Algumas variáveis, como nitrato, nitrogênio, ortofosfato e DBO, apresentaram valores mais elevados em comparação ao Cluster 1. Além disso, a diferença entre média e mediana em variáveis como amônia e DBO indicou assimetria positiva. Isso significa que a maioria das amostras possui valores mais moderados, mas existem algumas observações com valores altos que puxam a média para cima.

Esse padrão é muito importante porque conecta a clusterização com a análise exploratória anterior. Na etapa de assimetria, variáveis como amônia, DBO, ortofosfato e nitrato já haviam apresentado cauda longa à direita, indicando concentração de valores baixos com presença de valores extremos. O Cluster 0 parece capturar justamente essa zona de transição, onde a água ainda não está em condição crítica, mas já apresenta sinais claros de pressão ambiental.

O Cluster 2 foi o menor grupo, mas apresentou o comportamento mais severo. As médias de amônia e DBO foram extremamente elevadas. Mais importante ainda: as medianas também foram altas. Isso mostra que o resultado não foi causado apenas por poucos outliers isolados. O cluster inteiro apresenta um padrão crítico consistente.

Esse ponto é central para a interpretação da análise. Se apenas a média fosse alta, poderíamos suspeitar que poucos valores extremos estavam distorcendo o resultado. Porém, como a mediana também é elevada, conclui-se que a degradação é uma característica estrutural do Cluster 2.

Portanto, as estatísticas descritivas mostram que os três clusters representam perfis ambientais distintos:

- o Cluster 1 representa estabilidade e preservação;
- o Cluster 0 representa transição e atenção;
- o Cluster 2 representa degradação severa.

Essa estrutura é especialmente relevante porque foi encontrada sem o uso dos rótulos supervisionados, indicando que os próprios dados possuem uma organização ambiental coerente.

In [42]:
import plotly.express as px

# Criando rótulos interpretáveis
df['Cluster_Label'] = df['Cluster'].map({
    1: 'Adequada - Água mais preservada',
    0: 'Atenção - Condição intermediária',
    2: 'Crítica - Forte degradação ambiental'
})

# Variáveis utilizadas
features = [
    'Ammonia (mg/l)',
    'Biochemical Oxygen Demand (mg/l)',
    'Dissolved Oxygen (mg/l)',
    'Orthophosphate (mg/l)',
    'pH (ph units)',
    'Temperature (cel)',
    'Nitrogen (mg/l)',
    'Nitrate (mg/l)'
]

# Gerando os boxplots
for feature in features:

    fig = px.box(
        df,
        x='Cluster_Label',
        y=feature,
        color='Cluster_Label',
        points='outliers',
        title=f'Distribuição de {feature} por Cluster'
    )

    fig.update_layout(
        template='plotly_white',
        xaxis_title='Classificação do Cluster',
        yaxis_title=feature,
        showlegend=False,
        title_x=0.5
    )

    fig.show()

Output hidden; open in https://colab.research.google.com to view.

## Análise dos boxplots por cluster

Os boxplots foram utilizados para visualizar a distribuição das variáveis físico-químicas dentro de cada cluster. Essa visualização é importante porque permite analisar não apenas os valores médios, mas também a mediana, os quartis, a dispersão e a presença de outliers.

Em dados ambientais, essa etapa é fundamental. Muitas variáveis podem apresentar comportamento assimétrico, com grande concentração de valores baixos e poucos valores extremamente altos. Nesses casos, a média pode ser influenciada por extremos e não representar bem a maior parte das amostras.

Os boxplots permitiram observar que o Cluster 1 possui distribuições mais compactas para os principais parâmetros associados à qualidade da água, como amônia, DBO e nitrato. Isso indica que as amostras desse grupo possuem comportamento mais homogêneo e estável. Essa estabilidade reforça a classificação do cluster como Adequada.

No Cluster 0, os boxplots mostraram maior dispersão e maior presença de valores extremos. Esse comportamento é coerente com a interpretação de condição intermediária, pois o grupo reúne amostras que ainda apresentam características razoáveis, mas também contém observações com sinais mais fortes de degradação.

Essa variação interna é justamente o que caracteriza o Cluster 0 como uma zona de atenção. Ele não representa uma condição totalmente preservada, mas também não possui a severidade do Cluster 2. É um grupo de transição, onde a qualidade da água começa a apresentar alterações relevantes.

No Cluster 2, os boxplots evidenciaram valores muito elevados principalmente para amônia e DBO. Esses parâmetros são marcadores importantes de degradação, pois indicam possível carga orgânica elevada, decomposição de matéria orgânica e presença de compostos nitrogenados em concentrações críticas.

O principal insight dos boxplots é que a degradação não aparece de forma igual em todos os clusters. O Cluster 1 apresenta estabilidade, o Cluster 0 apresenta variabilidade e transição, enquanto o Cluster 2 apresenta um padrão extremo e consistente.

Assim, os boxplots confirmaram visualmente aquilo que as estatísticas descritivas já indicavam: os clusters não são apenas grupos matemáticos, mas representam perfis ambientais distintos e interpretáveis.

In [43]:
import pandas as pd

pca_df = pd.DataFrame(
    X_pca[:, :2],
    columns=['PC1', 'PC2']
)

pca_df['Cluster'] = clusters

pca_df['Cluster_Label'] = pca_df['Cluster'].map({
    1: 'Adequada',
    0: 'Atenção',
    2: 'Crítica'
})

In [44]:
import plotly.express as px

fig = px.scatter(
    pca_df,
    x='PC1',
    y='PC2',
    color='Cluster_Label',
    title='Visualização dos Clusters no Espaço PCA',
    opacity=0.7
)

fig.update_layout(
    template='plotly_white',
    xaxis_title='Componente Principal 1 (PC1)',
    yaxis_title='Componente Principal 2 (PC2)',
    title_x=0.5
)

fig.show()

Output hidden; open in https://colab.research.google.com to view.

## Visualização dos clusters no espaço PCA

O scatterplot do PCA foi utilizado para visualizar os clusters em duas dimensões, utilizando os dois primeiros componentes principais: PC1 e PC2.

Cada ponto no gráfico representa uma amostra do dataset. A cor do ponto indica o cluster ao qual essa amostra pertence. Os eixos PC1 e PC2 não representam variáveis individuais, como amônia, DBO ou nitrato. Eles representam combinações matemáticas das variáveis originais que capturam os principais padrões de variação dos dados.

Por isso, o objetivo desse gráfico não é interpretar diretamente “quanto maior o PC1, maior a amônia”, por exemplo. O objetivo é observar como as amostras se organizam no espaço reduzido pelo PCA e verificar se os clusters formados pelo K-Means possuem separação visual coerente.

O Cluster 1, classificado como Adequada, apareceu em uma região mais compacta do gráfico. Isso significa que as amostras desse grupo são mais próximas entre si no espaço PCA. Em termos ambientais, isso indica menor variabilidade físico-química e maior estabilidade. Essa leitura se conecta diretamente com as estatísticas descritivas e os boxplots, que também mostraram menor dispersão nesse cluster.

O Cluster 0, classificado como Atenção, apareceu mais espalhado, principalmente ao longo do eixo horizontal. Esse espalhamento indica maior variabilidade interna. Isso faz sentido porque esse cluster representa uma condição intermediária: parte das amostras ainda se aproxima do perfil adequado, enquanto outra parte já se aproxima de condições mais degradadas.

Essa característica é muito importante. O Cluster 0 funciona como uma região de transição entre a condição adequada e a condição crítica. Ambientalmente, isso representa um cenário em que a água já apresenta sinais de alteração, mas ainda não alcançou o padrão extremo observado no Cluster 2.

O Cluster 2, classificado como Crítica, apareceu em uma região mais afastada, principalmente ao longo do PC2. Isso mostra que as amostras críticas possuem uma assinatura estrutural diferente das demais. Mesmo sendo um grupo menor em quantidade, ele ocupa uma região própria no gráfico porque seus valores físico-químicos são muito distintos.

Esse resultado é um dos pontos mais fortes da análise. Visualmente, o gráfico mostra que as amostras críticas não estão simplesmente misturadas com as demais. Elas formam um padrão separado, o que reforça a ideia de que a degradação severa possui uma assinatura físico-química própria.

Outro ponto importante é que o tamanho visual ocupado por um cluster no gráfico não representa necessariamente a quantidade de amostras. Um cluster pode parecer maior porque está mais espalhado, e não porque possui mais dados. Esse é o caso do Cluster 0, que apresenta maior dispersão. Já o Cluster 1, mesmo concentrando muitas amostras, aparece mais compacto porque suas observações são mais semelhantes entre si.

Portanto, o scatterplot mostra uma narrativa ambiental muito clara:

- o Cluster 1 representa estabilidade;
- o Cluster 0 representa transição;
- o Cluster 2 representa criticidade.

A distribuição dos pontos confirma que o K-Means não separou os dados de forma aleatória. Os agrupamentos possuem coerência espacial, estatística e ambiental.

In [55]:
import pandas as pd
import plotly.express as px
from sklearn.preprocessing import MinMaxScaler

severity_features = [
    'Ammonia (mg/l)',
    'Biochemical Oxygen Demand (mg/l)',
    'Dissolved Oxygen (mg/l)',
    'pH (ph units)',
    'Nitrate (mg/l)'
]

severity_df = df[severity_features].copy()

scaler = MinMaxScaler()

severity_scaled = pd.DataFrame(
    scaler.fit_transform(severity_df),
    columns=severity_features,
    index=df.index
)

# Inverte OD: OD baixo = maior severidade
severity_scaled['Dissolved Oxygen (mg/l)'] = 1 - severity_scaled['Dissolved Oxygen (mg/l)']

# Severidade ambiental baseada nas variáveis do rótulo supervisionado
df['Environmental_Severity'] = severity_scaled.sum(axis=1)

bubble_df = pd.DataFrame({
    'PC1': X_pca[:, 0],
    'PC2': X_pca[:, 1],
    'Severity': df['Environmental_Severity'],
    'Cluster': df['Cluster_Label']
})

fig = px.scatter(
    bubble_df,
    x='PC1',
    y='PC2',
    size='Severity',
    color='Cluster',
    hover_data=['Severity'],
    title='Severidade Ambiental no Espaço PCA',
    opacity=0.75
)

fig.update_layout(
    template='plotly_white',
    title_x=0.5,
    width=1000,
    height=700,
    xaxis_title='PC1',
    yaxis_title='PC2'
)

fig.show()

Output hidden; open in https://colab.research.google.com to view.

## Interpretação do Bubble Chart — Severidade Ambiental no Espaço PCA

O `Bubble Chart` foi construído com o objetivo de adicionar uma nova camada de interpretação ambiental ao `scatterplot do PCA`.

Enquanto o scatterplot tradicional mostrava apenas:

- a posição das amostras no espaço PCA;
- a separação estrutural entre os clusters;
- e a distribuição dos grupos;

o Bubble Chart acrescenta uma nova dimensão visual:

> a intensidade da severidade ambiental.

Nesse gráfico:

- os eixos `PC1` e `PC2` representam os principais padrões estruturais encontrados pelo `PCA`;
- as cores representam os clusters identificados pelo `K-Means`;
- e o tamanho das bolhas representa um `score de severidade ambiental`.

Esse score foi construído utilizando as mesmas variáveis utilizadas na criação do rótulo supervisionado:

- `Amônia`
- `DBO`
- `Oxigênio Dissolvido`
- `pH`
- `Nitrato`

Essa decisão foi importante porque permitiu conectar diretamente:

```text
Clusterização → Critérios ambientais → Classificação supervisionada
```

As variáveis foram normalizadas para a mesma escala e combinadas em um índice único de severidade. Além disso, o Oxigênio Dissolvido foi invertido no cálculo, já que valores baixos de OD representam pior qualidade ambiental.

O resultado visual ficou extremamente coerente.

As menores bolhas concentram-se principalmente na região associada ao cluster Adequada, indicando amostras com menor severidade ambiental e melhores condições físico-químicas.

Já o cluster Atenção apresenta bolhas intermediárias e maior dispersão espacial, reforçando a ideia de uma zona de transição ambiental.

O comportamento mais marcante aparece no cluster Crítica, onde as bolhas tendem a crescer significativamente. Isso indica que as amostras mais severas ocupam regiões específicas do espaço PCA e apresentam assinatura físico-química própria.

Esse comportamento reforça um dos principais insights da análise:

a degradação ambiental não aparece como ruído aleatório.

Ela aparece como um padrão estrutural detectável.

Visualmente, o gráfico mostra uma progressão muito clara:

baixa severidade → transição → alta severidade

Isso torna o Bubble Chart uma das visualizações mais importantes da análise, porque ele consegue unir simultaneamente:

- estrutura matemática;
- separação estatística;
- intensidade ambiental;
- e interpretação visual.

No contexto do AquaSense, esse gráfico é especialmente relevante porque transforma padrões físico-químicos complexos em uma representação visual intuitiva da severidade ambiental, facilitando a interpretação dos diferentes estados de qualidade da água.

In [45]:
import pandas as pd
import plotly.express as px

# Médias por cluster
heatmap_data = df.groupby('Cluster_Label')[[
    'Ammonia (mg/l)',
    'Biochemical Oxygen Demand (mg/l)',
    'Dissolved Oxygen (mg/l)',
    'Orthophosphate (mg/l)',
    'pH (ph units)',
    'Temperature (cel)',
    'Nitrogen (mg/l)',
    'Nitrate (mg/l)'
]].mean()

# Heatmap
fig = px.imshow(
    heatmap_data,
    text_auto='.2f',
    aspect='auto',
    color_continuous_scale='RdYlBu_r',
    title='Heatmap das Médias dos Clusters'
)

fig.update_layout(
    template='plotly_white',
    title_x=0.5
)

fig.show()

## Interpretação do heatmap das médias dos clusters

O heatmap das médias foi utilizado para comparar visualmente o comportamento médio das variáveis físico-químicas em cada cluster. Cada linha representa um cluster e cada coluna representa uma variável.

As cores indicam a intensidade dos valores médios. Tons mais frios indicam valores menores, enquanto tons mais quentes indicam valores maiores. Dessa forma, o heatmap permite identificar rapidamente quais parâmetros mais caracterizam cada grupo.

No Cluster 1, classificado como Adequada, observa-se um padrão de valores baixos para amônia, DBO, nitrato, nitrogênio e ortofosfato. Ao mesmo tempo, o oxigênio dissolvido permanece elevado. Esse conjunto de características indica uma condição ambiental mais preservada, com menor carga orgânica, menor concentração de nutrientes e boa oxigenação.

No Cluster 0, classificado como Atenção, o heatmap mostra elevação principalmente em nitrato, nitrogênio, ortofosfato e DBO. Esse padrão sugere aumento de nutrientes no sistema, o que pode estar relacionado a pressões antrópicas, como escoamento superficial, lançamento de efluentes ou influência agrícola. Esse cluster não apresenta os extremos observados no grupo crítico, mas já mostra sinais claros de alteração ambiental.

No Cluster 2, classificado como Crítica, os valores de DBO e amônia se destacam fortemente. A DBO elevada indica grande carga orgânica biodegradável, enquanto a amônia elevada pode indicar presença de matéria orgânica nitrogenada, decomposição ou lançamento recente de efluentes. Esses dois parâmetros aparecem como os principais marcadores da degradação severa nesse grupo.

Um insight importante do heatmap é que a criticidade do Cluster 2 não está associada principalmente ao nitrato. Na verdade, o nitrato é mais elevado no Cluster 0. Isso mostra que os tipos de alteração ambiental não são iguais entre os grupos. O Cluster 0 parece estar mais relacionado ao aumento de nutrientes, enquanto o Cluster 2 está mais relacionado à carga orgânica e à amônia.

Essa diferença é relevante porque mostra que a degradação ambiental não ocorre por um único caminho. Diferentes combinações de parâmetros podem indicar diferentes formas de pressão sobre a qualidade da água.

O heatmap, portanto, sintetiza a assinatura química de cada cluster:

| Cluster | Assinatura predominante |
|---|---|
| Adequada | baixos poluentes e boa oxigenação |
| Atenção | aumento de nutrientes e condição intermediária |
| Crítica | DBO e amônia extremamente elevadas |

Essa visualização consolida a interpretação construída nas etapas anteriores. As estatísticas descritivas mostraram os valores, os boxplots mostraram a distribuição, o scatterplot mostrou a separação espacial e o heatmap mostrou a assinatura química de cada grupo.

# Interpretação dos Resultados da Clusterização

## Visão geral da análise

A etapa de clusterização foi utilizada com o objetivo de identificar padrões naturais nos dados físico-químicos da água, sem utilizar previamente os rótulos criados para o modelo supervisionado.

Diferente do aprendizado supervisionado, em que as amostras já estavam classificadas em categorias como **Adequada**, **Atenção** e **Crítica**, a clusterização buscou encontrar agrupamentos apenas com base na estrutura dos dados, considerando a similaridade entre as amostras.

Essa abordagem foi importante porque permitiu verificar se os próprios dados apresentavam, de forma natural, padrões semelhantes às classes ambientais utilizadas no modelo supervisionado.

---

## Padronização dos dados

Antes da aplicação do PCA e do K-Means, os dados foram padronizados.

Essa etapa foi necessária porque as variáveis físico-químicas possuem escalas muito diferentes. Por exemplo, o pH varia em uma faixa pequena, enquanto variáveis como DBO, amônia, nitrato e nitrogênio podem apresentar amplitudes muito maiores.

Sem padronização, variáveis com valores numericamente maiores poderiam dominar os cálculos de distância do K-Means, influenciando indevidamente a formação dos clusters.

Com a padronização, todas as variáveis passaram a contribuir de forma mais equilibrada para a análise.

---

## Aplicação do PCA

Após a padronização, foi aplicado o PCA, ou Análise de Componentes Principais.

O PCA foi utilizado para reduzir a dimensionalidade dos dados e transformar as variáveis originais em componentes principais, que representam combinações matemáticas dos parâmetros físico-químicos.

As variáveis originais analisadas foram:

- Amônia;
- DBO;
- Oxigênio Dissolvido;
- Ortofosfato;
- pH;
- Temperatura;
- Nitrogênio;
- Nitrato.

É importante destacar que os componentes principais não representam variáveis isoladas. O PC1, por exemplo, não corresponde diretamente à amônia ou ao pH. Ele representa uma combinação das variáveis que captura uma direção importante de variação dos dados.

Ou seja, o PCA procura identificar quais combinações de variáveis melhor explicam as diferenças entre as amostras.

---

## Variância explicada e acumulada

A análise da variância explicada mostrou que os dados ambientais não possuem um único padrão dominante. A variabilidade está distribuída entre vários componentes, o que é coerente com a complexidade de dados físico-químicos reais.

Os primeiros componentes explicaram partes relevantes da estrutura dos dados, e a variância acumulada indicou que, com seis componentes principais, foi possível preservar aproximadamente 88% da variabilidade total.

Isso significa que a maior parte dos padrões presentes nas oito variáveis originais foi mantida, ao mesmo tempo em que houve redução de dimensionalidade e simplificação da estrutura dos dados.

Essa escolha foi importante porque permitiu manter informação suficiente para a clusterização, evitando tanto a perda excessiva de padrões quanto a manutenção desnecessária de ruídos.

---

## Definição da quantidade de clusters

Para definir o número de clusters, foram utilizados dois métodos principais:

- Elbow Method;
- Silhouette Score.

O Elbow Method avaliou o WCSS, que mede o quanto os pontos estão dispersos dentro dos clusters. A curva apresentou uma redução mais forte até aproximadamente 3 ou 4 clusters, indicando que, a partir desse ponto, adicionar novos grupos trazia ganhos cada vez menores.

Em seguida, o Silhouette Score foi utilizado para avaliar a qualidade dos agrupamentos. Essa métrica considera tanto a coesão interna dos clusters quanto a separação entre eles.

Os resultados mostraram que:

- K=2 apresentou o maior Silhouette Score;
- K=3 ainda apresentou boa qualidade de separação;
- K=4 em diante começou a apresentar perda de consistência;
- K=5 e K=6 indicaram agrupamentos mais fracos e possivelmente artificiais.

Apesar de K=2 apresentar o melhor resultado matemático, a escolha por K=3 foi mais adequada para o contexto ambiental do projeto.

Essa decisão se justifica porque três clusters permitem representar melhor uma gradação ambiental:

- condição adequada;
- condição de atenção;
- condição crítica.

Além disso, essa estrutura se conecta diretamente com o modelo supervisionado, que também utilizou três categorias de qualidade: **Adequada**, **Atenção** e **Crítica**.

---

## Resultado geral dos clusters

Após a aplicação do K-Means com três grupos, os clusters foram interpretados da seguinte forma:

| Cluster | Interpretação |
|---|---|
| Cluster 1 | Adequada — Água mais preservada |
| Cluster 0 | Atenção — Condição intermediária |
| Cluster 2 | Crítica — Forte degradação ambiental |

Essa interpretação foi feita com base nas médias, medianas, dispersões, boxplots, scatterplot do PCA e heatmap das médias dos clusters.

---

## Cluster 1 — Água mais preservada

O Cluster 1 apresentou o maior número de amostras e representou o grupo com melhor condição físico-química geral.

Esse cluster apresentou:

- baixa amônia;
- baixa DBO;
- nitrato reduzido;
- oxigênio dissolvido elevado;
- pH dentro da faixa esperada;
- menor variabilidade interna.

A amônia apresentou média baixa e mediana ainda menor, indicando que a maior parte das amostras possui concentrações muito reduzidas. A DBO também ficou abaixo do limite de referência, sugerindo menor presença de matéria orgânica biodegradável.

O oxigênio dissolvido permaneceu elevado, indicando boa oxigenação da água. O nitrato também apresentou valores reduzidos, reforçando a ideia de menor pressão por nutrientes.

Além disso, o scatterplot do PCA mostrou que esse cluster é mais compacto, o que indica maior homogeneidade entre as amostras.

Assim, o Cluster 1 pode ser interpretado como o grupo de águas mais estáveis, preservadas e ambientalmente adequadas.

---

## Cluster 0 — Condição intermediária ou atenção

O Cluster 0 apresentou comportamento intermediário, com sinais de degradação parcial.

Esse grupo apresentou:

- amônia moderada;
- DBO próxima ou acima do limite;
- nitrato elevado;
- nitrogênio elevado;
- ortofosfato mais alto;
- maior dispersão entre as amostras.

A diferença entre média e mediana em variáveis como amônia e DBO indicou forte assimetria positiva, ou seja, concentração de valores mais baixos com presença de valores extremos puxando a média para cima.

Esse comportamento já havia sido identificado anteriormente na análise de assimetria, em que variáveis como amônia, DBO, ortofosfato e nitrato apresentaram cauda longa à direita.

O Cluster 0, portanto, parece representar uma zona de transição ambiental. Parte das amostras ainda apresenta condições razoáveis, enquanto outra parte já demonstra sinais importantes de comprometimento.

O nitrato elevado é um dos principais indicativos desse grupo, sugerindo maior presença de nutrientes e possível influência de fontes antrópicas, como esgoto, agricultura ou escoamento superficial.

Por isso, esse cluster se relaciona bem com a categoria **Atenção**.

---

## Cluster 2 — Forte degradação ambiental

O Cluster 2 foi o menor grupo em quantidade de amostras, mas o mais extremo em termos físico-químicos.

Esse cluster apresentou:

- amônia extremamente elevada;
- DBO extremamente elevada;
- forte distância estrutural em relação aos demais clusters;
- comportamento crítico bem definido.

Diferente do Cluster 0, no Cluster 2 a média e a mediana de amônia e DBO ficaram muito próximas e muito elevadas. Isso indica que não se trata apenas de poucos outliers puxando a média para cima. O grupo como um todo apresenta valores severamente degradados.

A DBO muito alta indica forte carga orgânica, enquanto a amônia elevada aponta para condições de degradação e possível presença de matéria orgânica nitrogenada ou lançamento recente de efluentes.

Esse comportamento confirma que o Cluster 2 representa um padrão crítico real, mesmo sendo composto por uma quantidade menor de amostras.

No scatterplot do PCA, esse cluster apareceu em uma região mais afastada, especialmente ao longo do PC2, mostrando que as amostras críticas possuem assinatura físico-química própria.

Esse resultado é muito relevante porque mostra que, mesmo sem utilizar os rótulos supervisionados, o K-Means conseguiu identificar naturalmente um grupo com forte degradação ambiental.

---

## Comparação com o modelo supervisionado

A comparação entre a clusterização e o modelo supervisionado é uma das partes mais importantes da análise.

No modelo supervisionado, os rótulos foram construídos a partir de cinco parâmetros relacionados à resolução CONAMA:

- pH;
- DBO;
- Oxigênio Dissolvido;
- Nitrato;
- Amônia.

A classificação supervisionada separou as amostras em:

- Adequada;
- Atenção;
- Crítica.

A classe adequada concentrou a maior parte dos dados, a classe atenção representou uma parcela intermediária e a classe crítica apresentou frequência muito baixa.

Na clusterização, a lógica foi diferente. O K-Means não recebeu os rótulos e não conhecia as regras da CONAMA. Ele agrupou as amostras apenas com base em similaridade matemática, considerando todas as variáveis utilizadas na análise.

Mesmo assim, os três clusters encontrados apresentaram uma estrutura muito próxima das categorias supervisionadas.

Isso mostra que os dados possuem uma organização interna coerente com os critérios ambientais utilizados no projeto.

---

## Diferenças entre clusterização e classificação supervisionada

As pequenas diferenças entre a distribuição dos clusters e a distribuição das classes supervisionadas são esperadas.

Isso ocorre porque os dois métodos funcionam de formas diferentes.

No modelo supervisionado, as classes foram definidas a partir de regras explícitas baseadas em cinco parâmetros. Já na clusterização, o algoritmo considerou padrões multivariados e relações entre todas as variáveis disponíveis, incluindo parâmetros que não entraram diretamente na construção do rótulo supervisionado.

Portanto, a clusterização não tinha obrigação de reproduzir exatamente as proporções das classes supervisionadas.

Na verdade, o fato de os resultados serem parecidos, mas não idênticos, é positivo. Isso indica que o K-Means encontrou padrões próprios nos dados, e não apenas uma repetição artificial da classificação anterior.

---

## Relação com a classe crítica

Um ponto importante observado no modelo supervisionado foi a dificuldade em aprender a classe crítica, devido ao forte desbalanceamento dos dados.

A classe crítica possuía poucas amostras, o que dificultou o aprendizado dos modelos supervisionados, como o Random Forest. Como essa classe era muito rara, o modelo tendia a favorecer as classes majoritárias.

Na clusterização, entretanto, o comportamento foi diferente.

Mesmo sendo um grupo menor, o Cluster 2 foi identificado porque suas amostras eram muito distintas do restante do dataset. A amônia e a DBO apresentaram valores extremamente elevados, criando uma assinatura físico-química suficientemente forte para que o K-Means separasse esse grupo.

Isso mostra uma diferença importante entre os métodos:

- o supervisionado sofre mais com classes raras quando há desbalanceamento;
- o não supervisionado pode identificar grupos pequenos quando eles são estruturalmente muito diferentes.

Esse é um dos principais achados da análise.

---

## Relação com a Resolução CONAMA

Ao comparar os clusters com os limites utilizados da CONAMA, observa-se uma coerência ambiental clara.

O Cluster 1 apresentou valores majoritariamente dentro dos limites considerados adequados, especialmente para DBO, nitrato, pH, oxigênio dissolvido e amônia.

O Cluster 0 apresentou sinais de atenção, com DBO próxima ou acima do limite e nitrato elevado. Isso indica um estado intermediário de qualidade, em que a água ainda apresenta bons sinais em alguns parâmetros, mas já demonstra pressões ambientais relevantes.

O Cluster 2 apresentou forte extrapolação dos limites, principalmente em amônia e DBO, caracterizando um cenário de degradação severa.

Assim, a estrutura dos clusters está alinhada com os critérios ambientais adotados no projeto.

---

## Interpretação dos gráficos

Os boxplots ajudaram a visualizar a distribuição dos parâmetros em cada cluster, mostrando mediana, dispersão e presença de outliers.

Eles confirmaram que:

- o Cluster 1 possui distribuições mais compactas;
- o Cluster 0 apresenta maior variabilidade e assimetria;
- o Cluster 2 possui valores extremos consistentes em amônia e DBO.

O scatterplot do PCA mostrou a separação visual entre os grupos. O Cluster 1 apareceu mais compacto, indicando estabilidade. O Cluster 0 apareceu mais espalhado, representando transição ambiental. O Cluster 2 apareceu mais afastado, especialmente no eixo PC2, reforçando sua assinatura crítica.

O heatmap das médias consolidou visualmente os perfis dos clusters. Ele mostrou claramente que:

- o Cluster 1 possui baixos valores para os principais poluentes;
- o Cluster 0 apresenta elevação de nutrientes;
- o Cluster 2 concentra os maiores valores de amônia e DBO.

---

## Síntese interpretativa

A clusterização revelou que os dados contam uma história ambiental bem definida.

A maior parte das amostras pertence a um perfil de água mais preservada, com baixa carga orgânica, baixa concentração de nutrientes e boa oxigenação.

Um segundo grupo representa uma condição intermediária, marcada por aumento de nutrientes, maior variabilidade e sinais de pressão ambiental. Esse grupo funciona como uma zona de transição entre a condição adequada e a condição crítica.

Por fim, um terceiro grupo, menor e mais extremo, representa forte degradação ambiental, caracterizada principalmente por valores muito elevados de amônia e DBO.

Essa estrutura mostra que a degradação ambiental não aparece de forma aleatória nos dados. Ela segue padrões físico-químicos claros, detectáveis mesmo sem o uso de rótulos supervisionados.

O principal insight da análise é que a clusterização conseguiu identificar naturalmente uma estrutura muito semelhante à classificação supervisionada, reforçando a coerência dos rótulos criados e a validade ambiental do projeto.

---

## Conclusão

A análise de clusterização mostrou que os dados físico-químicos possuem padrões ambientais consistentes e interpretáveis.

A utilização de PCA permitiu reduzir a dimensionalidade mantendo a maior parte da estrutura dos dados. O Elbow Method e o Silhouette Score apoiaram a escolha de três clusters, que se mostraram coerentes tanto do ponto de vista estatístico quanto ambiental.

A interpretação dos clusters indicou três perfis principais:

- água mais preservada;
- condição intermediária de atenção;
- forte degradação ambiental.

Esses perfis se conectam diretamente com as classes utilizadas no modelo supervisionado, demonstrando que os dados apresentam uma organização interna compatível com os critérios ambientais baseados na resolução CONAMA.

Assim, a clusterização não apenas complementou o modelo supervisionado, mas também fortaleceu a interpretação dos resultados, mostrando que os padrões ambientais encontrados são consistentes, estruturados e explicáveis.